# 03 · Model training and evaluation
Time-based split, baseline vs tree models, metrics and explanations. The final model is produced by `python run_pipeline.py`; this notebook inspects and re-runs the comparison.

In [ ]:
import sys, json
sys.path.insert(0, "..")          # make `src` importable from /notebooks
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.options.display.float_format = "{:,.2f}".format
from src import db, train_model
from src.feature_engineering import FEATURES, build_features

In [ ]:
amen = pd.read_csv(config.AMENITIES_CSV)
data = build_features(db.load_training_frame(), amen)       # stateless features -> safe before splitting
train, val, test = train_model.time_split(data)
print(len(train), len(val), len(test), "| test period:", test.sale_date.min().date(), "->", test.sale_date.max().date())

## Quick re-run of one model (fit on train only, evaluate on validation)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
pipe = train_model.make_pipeline(GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42))
pipe.fit(train[FEATURES], train.sale_price)
train_model.regression_metrics(val.sale_price, pipe.predict(val[FEATURES]))

## Saved comparison (CV on train, validation, test)

In [ ]:
metrics = json.load(open(config.METRICS_PATH))
print("best:", metrics["best_model"], "| interval coverage on test:", round(metrics["interval"]["test_coverage"], 3))
pd.DataFrame(metrics["models"]).set_index("model")

## Actual vs predicted and residuals

In [ ]:
p = pd.read_csv(config.TEST_PRED_PATH)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(p.actual, p.predicted, s=6, alpha=.5); lim = [p.actual.min(), p.actual.max()]; ax[0].plot(lim, lim, "r--")
ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted"); ax[0].set_title("Actual vs predicted")
ax[1].scatter(p.predicted, p.residual, s=6, alpha=.5, color="#8e6bd8"); ax[1].axhline(0, color="r", ls="--"); ax[1].set_title("Residuals")
plt.tight_layout(); plt.show()

## Feature importance (SHAP if installed, otherwise permutation)

In [ ]:
imp = json.load(open(config.IMPORTANCE_PATH)); print("method:", imp["method"])
g = pd.DataFrame(imp["groups"]).set_index("group")["share_pct"].sort_values()
g.plot.barh(figsize=(6, 3.5), color="#2f6fe4"); plt.xlabel("share of total importance (%)"); plt.show()